# BigSmall Multitask Training Notebook

This notebook trains the BigSmall multitask model for rPPG, respiration, and AU estimation.
import csv
import time
import math
import glob


In [ ]:
import os
import sys
import json
import glob
import pickle
import shutil
import numpy as np
import pandas as pd
import cv2
from tqdm import tqdm
import torch
import torch.multiprocessing
torch.multiprocessing.set_sharing_strategy('file_system')
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

REPO_ROOT = "d:/New folder/Non-Invasive/rPPG"


## Inlined source

The cells below contain the model / loss source that was previously imported from the `neural_methods/` (and `evaluation/`) packages. They are inlined here so the notebook is self-contained.


In [ ]:
# === inlined from neural_methods/model/BigSmall.py ===
"""BigSmall: Multitask Network for AU / Respiration / PPG

BigSmall: Efficient Multi-Task Learning
For Physiological Measurements
Girish Narayanswamy, Yujia (Nancy) Liu, Yuzhe Yang, Chengqian (Jack) Ma, 
Xin Liu, Daniel McDuff, Shwetak Patel

https://arxiv.org/abs/2303.11573
"""

import torch
import torch.nn as nn


#####################################################
############ Wrapping Time Shift Module #############
#####################################################
class WTSM(nn.Module):
    def __init__(self, n_segment=3, fold_div=3):
        super(WTSM, self).__init__()
        self.n_segment = n_segment
        self.fold_div = fold_div

    def forward(self, x):
        nt, c, h, w = x.size()
        n_batch = nt // self.n_segment
        x = x.view(n_batch, self.n_segment, c, h, w)
        fold = c // self.fold_div
        out = torch.zeros_like(x)
        out[:, :-1, :fold] = x[:, 1:, :fold]  # shift left
        out[:, -1, :fold] = x[:, 0, :fold] # wrap left
        out[:, 1:, fold: 2 * fold] = x[:, :-1, fold: 2 * fold]  # shift right
        out[:, 0, fold: 2 * fold] = x[:, -1, fold: 2 * fold]  # wrap right
        out[:, :, 2 * fold:] = x[:, :, 2 * fold:]  # no shift for final fold
        return out.view(nt, c, h, w)



#######################################################################################
##################################### BigSmall Model ##################################
#######################################################################################
class BigSmall(nn.Module):

    def __init__(self, in_channels=3, nb_filters1=32, nb_filters2=64, kernel_size=3, 
                 dropout_rate1=0.25, dropout_rate2=0.5, dropout_rate3=0.5, pool_size1=(2, 2), pool_size2=(4,4),
                 nb_dense=128, out_size_bvp=1, out_size_resp=1, out_size_au=12, n_segment=3):

        super(BigSmall, self).__init__()

        self.in_channels = in_channels
        self.kernel_size = kernel_size
        self.dropout_rate1 = dropout_rate1
        self.dropout_rate2 = dropout_rate2
        self.dropout_rate3 = dropout_rate3
        self.pool_size1 = pool_size1
        self.pool_size2 = pool_size2
        self.nb_filters1 = nb_filters1
        self.nb_filters2 = nb_filters2
        self.nb_dense = nb_dense

        self.out_size_bvp = out_size_bvp
        self.out_size_resp = out_size_resp
        self.out_size_au = out_size_au

        self.n_segment = n_segment

        # Big Convolutional Layers
        self.big_conv1 = nn.Conv2d(self.in_channels, self.nb_filters1, kernel_size=self.kernel_size, padding=(1, 1), bias=True)
        self.big_conv2 = nn.Conv2d(self.nb_filters1, self.nb_filters1, kernel_size=self.kernel_size, padding=(1, 1), bias=True)
        self.big_conv3 = nn.Conv2d(self.nb_filters1, self.nb_filters1, kernel_size=self.kernel_size, padding=(1, 1), bias=True)
        self.big_conv4 = nn.Conv2d(self.nb_filters1, self.nb_filters2, kernel_size=self.kernel_size, padding=(1, 1), bias=True)
        self.big_conv5 = nn.Conv2d(self.nb_filters2, self.nb_filters2, kernel_size=self.kernel_size, padding=(1, 1), bias=True)
        self.big_conv6 = nn.Conv2d(self.nb_filters2, self.nb_filters2, kernel_size=self.kernel_size, padding=(1, 1), bias=True)

        # Big Avg Pooling / Dropout Layers
        self.big_avg_pooling1 = nn.AvgPool2d(self.pool_size1)
        self.big_dropout1 = nn.Dropout(self.dropout_rate1)
        self.big_avg_pooling2 = nn.AvgPool2d(self.pool_size1)
        self.big_dropout2 = nn.Dropout(self.dropout_rate2)
        self.big_avg_pooling3 = nn.AvgPool2d(self.pool_size2)
        self.big_dropout3 = nn.Dropout(self.dropout_rate3)

        # TSM layers
        self.TSM_1 = WTSM(n_segment=self.n_segment)
        self.TSM_2 = WTSM(n_segment=self.n_segment)
        self.TSM_3 = WTSM(n_segment=self.n_segment)
        self.TSM_4 = WTSM(n_segment=self.n_segment)
        
        # Small Convolutional Layers
        self.small_conv1 = nn.Conv2d(self.in_channels, self.nb_filters1, kernel_size=self.kernel_size, padding=(1,1), bias=True)
        self.small_conv2 = nn.Conv2d(self.nb_filters1, self.nb_filters1, kernel_size=self.kernel_size, padding=(1,1), bias=True)
        self.small_conv3 = nn.Conv2d(self.nb_filters1, self.nb_filters1, kernel_size=self.kernel_size, padding=(1,1), bias=True)
        self.small_conv4 = nn.Conv2d(self.nb_filters1, self.nb_filters2, kernel_size=self.kernel_size, padding=(1,1), bias=True)

        # AU Fully Connected Layers 
        self.au_fc1 = nn.Linear(5184, self.nb_dense, bias=True)
        self.au_fc2 = nn.Linear(self.nb_dense, self.out_size_au, bias=True)

        # BVP Fully Connected Layers 
        self.bvp_fc1 = nn.Linear(5184, self.nb_dense, bias=True)
        self.bvp_fc2 = nn.Linear(self.nb_dense, self.out_size_bvp, bias=True)

        # Resp Fully Connected Layers 
        self.resp_fc1 = nn.Linear(5184, self.nb_dense, bias=True)
        self.resp_fc2 = nn.Linear(self.nb_dense, self.out_size_resp, bias=True)


    def forward(self, inputs, params=None):

        big_input = inputs[0] # big res 
        small_input = inputs[1] # small res

        # reshape Big 
        nt, c, h, w = big_input.size()
        n_batch = nt // self.n_segment
        big_input = big_input.view(n_batch, self.n_segment, c, h, w)
        big_input = torch.moveaxis(big_input, 1, 2) # color channel to idx 1, sequence channel to idx 2
        big_input = big_input[:, :, 0, :, :] # use only first frame in sequences 


        # Big Conv block 1
        b1 = nn.functional.relu(self.big_conv1(big_input))
        b2 = nn.functional.relu(self.big_conv2(b1))
        b3 = self.big_avg_pooling1(b2)
        b4 = self.big_dropout1(b3)

        # Big Conv block 2
        b5 = nn.functional.relu(self.big_conv3(b4))
        b6 = nn.functional.relu(self.big_conv4(b5))
        b7 = self.big_avg_pooling2(b6)
        b8 = self.big_dropout2(b7)

        # Big Conv block 3
        b9 = nn.functional.relu(self.big_conv5(b8))
        b10 = nn.functional.relu(self.big_conv6(b9))
        b11 = self.big_avg_pooling3(b10)
        b12 = self.big_dropout3(b11)

        # Reformat Big Shape For Concat w/ Small Branch
        b13 = torch.stack((b12, b12, b12), 2) #TODO: this is hardcoded for num_segs = 3: change this...
        b14 = torch.moveaxis(b13, 1, 2)
        bN, bD, bC, bH, bW = b14.size()
        b15 = b14.reshape(int(bN*bD), bC, bH, bW)

        # Small Conv block 1
        s1 = self.TSM_1(small_input)
        s2 = nn.functional.relu(self.small_conv1(s1))
        s3 = self.TSM_2(s2)
        s4 = nn.functional.relu(self.small_conv2(s3))

        # Small Conv block 2
        s5 = self.TSM_3(s4)
        s6 = nn.functional.relu(self.small_conv3(s5))
        s7 = self.TSM_4(s6)
        s8 = nn.functional.relu(self.small_conv4(s7))

        # Shared Layers
        concat = b15 + s8 # sum layers

        # share1 = concat.view(concat.size(0), -1) # flatten entire tensors
        share1 = concat.reshape(concat.size(0), -1)

        # AU Output Layers
        aufc1 = nn.functional.relu(self.au_fc1(share1))
        au_out = self.au_fc2(aufc1)

        # BVP Output Layers
        bvpfc1 = nn.functional.relu(self.bvp_fc1(share1))
        bvp_out = self.bvp_fc2(bvpfc1)

        # Resp Output Layers
        respfc1 = nn.functional.relu(self.resp_fc1(share1))
        resp_out = self.resp_fc2(respfc1)

        return au_out, bvp_out, resp_out




## Training utilities

Seed, train/val split, HR-MAE, best-checkpoint saver, early-stopping, CSV logger.


In [ ]:
# === Training utilities (shared across notebooks) ===
# seed, train/val split, HR-MAE, best-checkpoint saver, early-stopping, CSV logger.
import os
import random as _random
import csv as _csv
import time as _time
import numpy as _np
import torch as _torch
from scipy.signal import periodogram as _periodogram


def set_seed(seed: int = 42):
    """Set seeds for reproducibility across random, numpy, torch (CPU + CUDA)."""
    _random.seed(seed)
    _np.random.seed(seed)
    _torch.manual_seed(seed)
    _torch.cuda.manual_seed_all(seed)
    _torch.backends.cudnn.deterministic = True
    _torch.backends.cudnn.benchmark = False


def train_val_split(dataset, val_ratio: float = 0.2, seed: int = 42):
    """Random split returning (train_subset, val_subset) with a seeded generator."""
    n_total = len(dataset)
    n_val = max(1, int(n_total * val_ratio))
    n_train = n_total - n_val
    g = _torch.Generator().manual_seed(seed)
    return _torch.utils.data.random_split(dataset, [n_train, n_val], generator=g)


def compute_hr_fft(signal_1d, fps: int = 30, lo_hz: float = 0.6, hi_hz: float = 3.3) -> float:
    """Peak-frequency HR (bpm) of a 1-D signal via periodogram, restricted to a band."""
    sig = signal_1d.detach().cpu().numpy() if _torch.is_tensor(signal_1d) else _np.asarray(signal_1d)
    sig = sig.astype(_np.float64).ravel()
    if sig.size < 8 or sig.std() < 1e-8:
        return 0.0
    sig = sig - sig.mean()
    freqs, psd = _periodogram(sig, fs=fps)
    band = (freqs >= lo_hz) & (freqs <= hi_hz)
    if not band.any():
        return 0.0
    return float(freqs[band][psd[band].argmax()] * 60.0)


def compute_hr_mae_batch(preds, labels, fps: int = 30) -> float:
    """Mean absolute HR error (bpm) over a batch.  preds/labels: (N, T) or (T,)."""
    if preds.dim() == 1:
        preds = preds.unsqueeze(0)
    if labels.dim() == 1:
        labels = labels.unsqueeze(0)
    errs = []
    for i in range(preds.shape[0]):
        hr_p = compute_hr_fft(preds[i], fps)
        hr_l = compute_hr_fft(labels[i], fps)
        errs.append(abs(hr_p - hr_l))
    return float(_np.mean(errs)) if errs else 0.0


class BestCheckpointSaver:
    """Save the model's state_dict whenever a tracked metric improves."""
    def __init__(self, path: str, mode: str = "min"):
        assert mode in ("min", "max")
        self.path = path
        self.mode = mode
        self.best = float("inf") if mode == "min" else -float("inf")
        os.makedirs(os.path.dirname(os.path.abspath(path)) or ".", exist_ok=True)

    def step(self, model, metric: float) -> bool:
        improved = (metric < self.best) if self.mode == "min" else (metric > self.best)
        if improved and not (metric != metric):  # reject NaN
            self.best = metric
            _torch.save(model.state_dict(), self.path)
            return True
        return False


class EarlyStopping:
    """Stop training when the tracked metric stops improving for `patience` epochs."""
    def __init__(self, patience: int = 5, mode: str = "min", min_delta: float = 0.0):
        assert mode in ("min", "max")
        self.patience = patience
        self.mode = mode
        self.min_delta = min_delta
        self.best = float("inf") if mode == "min" else -float("inf")
        self.counter = 0
        self.should_stop = False

    def step(self, metric: float) -> bool:
        improved = (
            (self.mode == "min" and metric < self.best - self.min_delta)
            or (self.mode == "max" and metric > self.best + self.min_delta)
        )
        if improved:
            self.best = metric
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
        return self.should_stop


class MetricLogger:
    """Append per-epoch metrics to a CSV.  Creates the file with headers on init."""
    def __init__(self, csv_path: str, fieldnames=None):
        self.csv_path = csv_path
        self.fieldnames = list(fieldnames) if fieldnames else [
            "epoch", "train_loss", "val_loss", "val_hr_mae", "lr", "time_sec"
        ]
        os.makedirs(os.path.dirname(os.path.abspath(csv_path)) or ".", exist_ok=True)
        with open(csv_path, "w", newline="") as f:
            _csv.DictWriter(f, fieldnames=self.fieldnames).writeheader()

    def log(self, **row):
        with open(self.csv_path, "a", newline="") as f:
            _csv.DictWriter(f, fieldnames=self.fieldnames).writerow(
                {k: row.get(k, "") for k in self.fieldnames}
            )


# Seed the run for reproducibility
set_seed(42)
print("Training utils ready  |  seed=42  |  HR-MAE band: 0.6-3.3 Hz (36-198 bpm)")


In [ ]:
# ----- paths -----
RAW_DATA_PATH       = os.path.join(REPO_ROOT, "data/Headmotion")
PREPROCESSED_PATH   = os.path.join(REPO_ROOT, "preprocessed_data/Headmotion/bigsmall")

MODEL_SAVE_PATH     = os.path.join(REPO_ROOT, "final_model_release", "BigSmall_Multitask.pth")
os.makedirs(os.path.dirname(MODEL_SAVE_PATH), exist_ok=True)

# ----- video / signal params -----
VIDEO_FPS   = 30       # camera frame rate
PPG_FS      = 60       # PPG sensor sampling rate (Hz)

# ----- BigSmall preprocessing params -----
CHUNK_LENGTH = 3       # frames per clip (same as training config)
BIG_H, BIG_W = 144, 144
SMALL_H, SMALL_W = 9, 9

# ----- device -----
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

# ----- 49-channel label layout (matches BigSmallTrainer label_list) -----
LABEL_IDX_BVP  = 0   # bp_wave (green-channel PPG signal)
LABEL_IDX_HR   = 1   # HR_bpm
LABEL_IDX_RESP = 5   # resp_wave
NUM_LABEL_CH   = 49

os.makedirs(PREPROCESSED_PATH, exist_ok=True)
print("PREPROCESSED_PATH:", PREPROCESSED_PATH)
print("MODEL_SAVE_PATH:", MODEL_SAVE_PATH)


## 1. Preprocessing Tools (Optional)

These functions can be used to convert raw video and `.csv` ground truths into BigSmall's required multi-scale format.

In [ ]:
# Read video frames
def read_video_frames(video_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f"Cannot open video: {video_path}")

    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()

    if not frames:
        raise ValueError(f"Empty video: {video_path}")
    return np.stack(frames, axis=0)


In [ ]:
def read_ppg_synced(session_path, num_frames):
    import pandas as pd
    import numpy as np
    import os
    
    frame_df = pd.read_csv(os.path.join(session_path, "frame_timestamps.csv"))
    
    if len(frame_df) != num_frames:
        min_len = min(len(frame_df), num_frames)
        frame_t = frame_df["timestamp"].values[:min_len]
    else:
        frame_t = frame_df["timestamp"].values

    ppg_df = pd.read_csv(os.path.join(session_path, "ppg.csv"))
    ppg_t = ppg_df["Timestamp"].values
    ppg_val = ppg_df["PPG"].values
    
    frame_t_clipped = np.clip(frame_t, ppg_t[0], ppg_t[-1])
    ppg_resampled = np.interp(frame_t_clipped, ppg_t, ppg_val)
    
    return ppg_resampled.astype(np.float32)


In [ ]:
# Normalization functions
def diff_normalize_data(data):
    data = data.astype(np.float32)
    n, h, w, c = data.shape
    out = np.zeros_like(data)
    out[:n - 1] = (data[1:] - data[:-1]) / (data[1:] + data[:-1] + 1e-7)
    std = np.std(out)
    if std > 0:
        out /= std
    return out

def standardized_data(data):
    data = data.astype(np.float32)
    m = np.mean(data)
    s = np.std(data)
    if s > 0:
        data = (data - m) / s
    else:
        data = np.zeros_like(data)
    data = np.where(np.isnan(data), np.zeros_like(data), data)
    return data

def diff_normalize_label(label):
    diff = np.diff(label.astype(np.float64), axis=0)
    s = np.std(diff)
    if s > 0:
        diff = diff / s
    return np.append(diff, [0.0]).astype(np.float32)


In [ ]:
# Face crop + resize
def crop_face_resize(frames, out_h, out_w, large_box_coef=1.5):
    xml_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
    detector  = cv2.CascadeClassifier(xml_path)

    frame0 = frames[0]
    if frame0.dtype != np.uint8:
        frame0 = np.clip(frame0, 0, 255).astype(np.uint8)
    gray = cv2.cvtColor(frame0, cv2.COLOR_RGB2GRAY)

    faces = detector.detectMultiScale(gray, scaleFactor=1.3, minNeighbors=5)
    H, W = frames.shape[1], frames.shape[2]

    if len(faces) > 0:
        x, y, fw, fh = max(faces, key=lambda f: f[2])  # largest face
        x  = max(0, int(x  - (large_box_coef - 1.0) / 2.0 * fw))
        y  = max(0, int(y  - (large_box_coef - 1.0) / 2.0 * fh))
        fw = min(int(fw * large_box_coef), W - x)
        fh = min(int(fh * large_box_coef), H - y)
    else:
        x, y, fw, fh = 0, 0, W, H  # fallback: full frame

    C = frames.shape[3]
    resized = np.zeros((len(frames), out_h, out_w, C), dtype=np.float32)
    for i, frame in enumerate(frames):
        crop = frame[y : y + fh, x : x + fw]
        if crop.size == 0:
            crop = frame
        resized[i] = cv2.resize(crop.astype(np.float32), (out_w, out_h),
                                interpolation=cv2.INTER_AREA)
    return resized

def resize_frames(frames, out_h, out_w):
    T, H, W, C = frames.shape
    resized = np.zeros((T, out_h, out_w, C), dtype=np.float32)
    for i in range(T):
        resized[i] = cv2.resize(frames[i], (out_w, out_h),
                                interpolation=cv2.INTER_AREA)
    return resized


In [ ]:
# Discover subjects and preprocess (Set PREPROCESS_DATA = True to run)
all_dirs = sorted([
    d for d in glob.glob(os.path.join(RAW_DATA_PATH, "*"))
    if os.path.isdir(d) and os.path.basename(d) != "videos"
])
subjects = []
for subj_dir in all_dirs:
    subj_id  = os.path.basename(subj_dir)
    subj_key = subj_id.replace("_", "")
    video_pattern = os.path.join(RAW_DATA_PATH, "videos", f"{subj_id}.mkv")
    video_files = glob.glob(video_pattern)
    if not video_files:
        continue
    subjects.append({
        "subj_id":      subj_id,
        "subj_key":     subj_key,
        "video_path":   video_files[0],
        "session_path": subj_dir,
    })

# Set PREPROCESS_DATA = False if data is already prepared in PREPROCESSED_PATH
PREPROCESS_DATA = False 
if PREPROCESS_DATA:
    if os.path.exists(PREPROCESSED_PATH):
        shutil.rmtree(PREPROCESSED_PATH)
    os.makedirs(PREPROCESSED_PATH)

    for subj in subjects:
        subj_key     = subj["subj_key"]
        video_path   = subj["video_path"]
        session_path = subj["session_path"]
        
        frames = read_video_frames(video_path)
        T = frames.shape[0]
        ppg_signal = read_ppg_synced(session_path, T)

        labels = np.full((T, NUM_LABEL_CH), -1.0, dtype=np.float32)
        labels[:, LABEL_IDX_BVP] = ppg_signal
        
        frames_big = crop_face_resize(frames, BIG_H, BIG_W)
        big_data   = standardized_data(frames_big)
        diff_big   = diff_normalize_data(frames_big)
        small_data = resize_frames(diff_big, SMALL_H, SMALL_W)
        labels[:, LABEL_IDX_BVP] = diff_normalize_label(ppg_signal)

        clip_num    = T // CHUNK_LENGTH
        big_clips   = np.array([big_data[i*CHUNK_LENGTH:(i+1)*CHUNK_LENGTH]   for i in range(clip_num)])
        small_clips = np.array([small_data[i*CHUNK_LENGTH:(i+1)*CHUNK_LENGTH] for i in range(clip_num)])
        label_clips = np.array([labels[i*CHUNK_LENGTH:(i+1)*CHUNK_LENGTH]     for i in range(clip_num)])

        subj_dir = os.path.join(PREPROCESSED_PATH, subj_key)
        os.makedirs(subj_dir, exist_ok=True)

        for chunk_idx in range(clip_num):
            input_path = os.path.join(subj_dir, f"{subj_key}_input{chunk_idx}.pickle")
            label_path = os.path.join(subj_dir, f"{subj_key}_label{chunk_idx}.npy")
            frames_dict = {0: big_clips[chunk_idx], 1: small_clips[chunk_idx]}
            with open(input_path, "wb") as fh:
                pickle.dump(frames_dict, fh, protocol=pickle.HIGHEST_PROTOCOL)
            np.save(label_path, label_clips[chunk_idx])
            print(f"Processed {subj_key} - Chunk {chunk_idx}")


## 2. Dataset and DataLoader

In [ ]:
# Gather all files for DataLoader
all_input_files = []
for subj_dir in glob.glob(os.path.join(PREPROCESSED_PATH, "*")):
    if os.path.isdir(subj_dir):
        subj_files = glob.glob(os.path.join(subj_dir, "*_input*.pickle"))
        all_input_files.extend(subj_files)
print(f"Found {len(all_input_files)} preprocessed clips.")

class BigSmallDataset(Dataset):
    def __init__(self, input_files):
        self.inputs = sorted(input_files)
        self.labels = [
            f.replace("input", "label").replace(".pickle", ".npy")
            for f in self.inputs
        ]

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, index):
        with open(self.inputs[index], "rb") as fh:
            data = pickle.load(fh)

        # NDHWC -> NDCHW
        data[0] = np.float32(np.transpose(data[0], (0, 3, 1, 2)))
        data[1] = np.float32(np.transpose(data[1], (0, 3, 1, 2)))

        label = np.float32(np.load(self.labels[index]))  # (D, 49)

        fname      = os.path.basename(self.inputs[index])
        split_idx  = fname.index("_")
        subject_id = fname[:split_idx]                     # "S000"
        chunk_id   = fname[split_idx + 6:].split(".")[0]  # +6 skips "_input"

        return data, label, subject_id, chunk_id

def bigsmall_collate(batch):
    data_dicts, labels, subjects, chunk_ids = zip(*batch)

    data_big   = torch.stack([torch.from_numpy(d[0]) for d in data_dicts], dim=0)
    data_small = torch.stack([torch.from_numpy(d[1]) for d in data_dicts], dim=0)
    labels_t   = torch.stack([torch.from_numpy(l)    for l in labels],     dim=0)

    return data_big, data_small, labels_t, list(subjects), list(chunk_ids)

dataset = BigSmallDataset(all_input_files)

# Train/val split (seeded) + DataLoaders
SEED = 42
VAL_RATIO = 0.2
EPOCHS = 30
PATIENCE = 5

train_ds, val_ds = train_val_split(dataset, val_ratio=VAL_RATIO, seed=SEED)
g = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,
                          num_workers=4, pin_memory=True,
                          collate_fn=bigsmall_collate, generator=g)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False,
                          num_workers=2, pin_memory=True,
                          collate_fn=bigsmall_collate)
print(f"Train: {len(train_ds)} ({len(train_loader)} batches)  |  Val: {len(val_ds)} ({len(val_loader)} batches)")


## 3. Model, Loss, Optimizer

In [ ]:
# Training Setup
model = BigSmall(n_segment=CHUNK_LENGTH).to(DEVICE)

AU_weights = torch.as_tensor([9.64, 11.74, 16.77, 1.05, 0.53, 0.56, 
                              0.75, 0.69, 8.51, 6.94, 5.03, 25.00]).to(DEVICE)

criterionAU = nn.BCEWithLogitsLoss(pos_weight=AU_weights).to(DEVICE)
criterionBVP = nn.MSELoss().to(DEVICE)
criterionRESP = nn.MSELoss().to(DEVICE)

optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0)

# 49 channel list layout from trainer
label_list = ['bp_wave', 'HR_bpm', 'systolic_bp', 'diastolic_bp', 'mean_bp', 
              'resp_wave', 'resp_bpm', 'eda', 
              'AU01', 'AU02', 'AU04', 'AU05', 'AU06', 'AU06int', 'AU07', 'AU09', 'AU10', 'AU10int', 
              'AU11', 'AU12', 'AU12int', 'AU13', 'AU14', 'AU14int', 'AU15', 'AU16', 'AU17', 'AU17int', 
              'AU18', 'AU19', 'AU20', 'AU22', 'AU23', 'AU24', 'AU27', 'AU28', 'AU29', 'AU30', 'AU31', 
              'AU32', 'AU33', 'AU34', 'AU35', 'AU36', 'AU37', 'AU38', 'AU39',
              'pos_bvp','pos_env_norm_bvp']

au_labels = ['AU01', 'AU02', 'AU04', 'AU06', 'AU07', 'AU10', 'AU12', 'AU14', 'AU15', 'AU17', 'AU23', 'AU24']
LABEL_IDXS_AU = [label_list.index(l) for l in au_labels]

BASE_LEN = CHUNK_LENGTH


## 4. Inline Training Loop

In [ ]:
# Optimized BigSmall multi-task training (best-by-BVP-loss on val, grad-clip, CSV)
LOG_PATH = os.path.join(REPO_ROOT, "results/Headmotion/bigsmall/train_logs/BigSmall.csv")

saver   = BestCheckpointSaver(MODEL_SAVE_PATH, mode="min")
stopper = EarlyStopping(patience=PATIENCE, mode="min")
logger  = MetricLogger(LOG_PATH, fieldnames=[
    "epoch", "train_loss", "val_loss", "val_bvp_loss", "val_resp_loss",
    "val_au_loss", "lr", "time_sec",
])
# Use Adam (weight_decay=0 means AdamW = Adam anyway), or set wd=1e-2 for real AdamW
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=1e-4, epochs=EPOCHS, steps_per_epoch=len(train_loader)
)


def _forward_bigsmall(batch):
    data_big, data_small, labels_t, _, _ = batch
    N, D, C_b, H_b, W_b = data_big.shape
    big_flat   = data_big.view(N * D, C_b, H_b, W_b).to(DEVICE, non_blocking=True)
    small_flat = data_small.view(N * D, data_small.shape[2], data_small.shape[3], data_small.shape[4]).to(DEVICE, non_blocking=True)
    trim = (N * D) // BASE_LEN * BASE_LEN
    big_flat, small_flat = big_flat[:trim], small_flat[:trim]
    labels_flat = labels_t.view(N * D, NUM_LABEL_CH).to(DEVICE, non_blocking=True)[:trim]
    au_out, bvp_out, resp_out = model((big_flat, small_flat))
    target_bvp  = labels_flat[:, LABEL_IDX_BVP].unsqueeze(-1)
    target_resp = labels_flat[:, LABEL_IDX_RESP].unsqueeze(-1)
    target_au   = torch.clamp(labels_flat[:, LABEL_IDXS_AU], 0.0, 1.0)
    bvp_loss  = criterionBVP(bvp_out, target_bvp)
    resp_loss = criterionRESP(resp_out, target_resp)
    au_loss   = criterionAU(au_out, target_au)
    return bvp_loss, resp_loss, au_loss


for epoch in range(EPOCHS):
    t0 = time.time()
    model.train()
    train_loss_sum, n_train = 0.0, 0
    tbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [train]", ncols=90)
    for batch in tbar:
        optimizer.zero_grad()
        bvp_loss, resp_loss, au_loss = _forward_bigsmall(batch)
        loss = bvp_loss + resp_loss + au_loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        train_loss_sum += loss.item()
        n_train += 1
        tbar.set_postfix(loss=f"{loss.item():.4f}")
    train_loss = train_loss_sum / max(1, n_train)

    model.eval()
    val_loss_sum = val_bvp_sum = val_resp_sum = val_au_sum = 0.0
    n_val = 0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [val]  ", ncols=90):
            bvp_loss, resp_loss, au_loss = _forward_bigsmall(batch)
            val_bvp_sum  += bvp_loss.item()
            val_resp_sum += resp_loss.item()
            val_au_sum   += au_loss.item()
            val_loss_sum += (bvp_loss + resp_loss + au_loss).item()
            n_val += 1
    val_loss = val_loss_sum / max(1, n_val)
    val_bvp  = val_bvp_sum  / max(1, n_val)
    val_resp = val_resp_sum / max(1, n_val)
    val_au   = val_au_sum   / max(1, n_val)
    elapsed = time.time() - t0
    cur_lr = optimizer.param_groups[0]["lr"]

    # Save best by BVP loss (the primary rPPG signal)
    improved = saver.step(model, val_bvp)
    marker = " <- best (by BVP)" if improved else ""
    print(f"Epoch {epoch+1:2d}  train={train_loss:.4f}  val={val_loss:.4f}  "
          f"BVP={val_bvp:.4f}  RESP={val_resp:.4f}  AU={val_au:.4f}  "
          f"lr={cur_lr:.2e}  ({elapsed:.1f}s){marker}")
    logger.log(epoch=epoch+1, train_loss=train_loss, val_loss=val_loss,
               val_bvp_loss=val_bvp, val_resp_loss=val_resp, val_au_loss=val_au,
               lr=cur_lr, time_sec=elapsed)

    if stopper.step(val_bvp):
        print(f"Early stopping at epoch {epoch+1}")
        break

print(f"\nBest val BVP loss: {saver.best:.4f}  ->  {MODEL_SAVE_PATH}")
